# 电信推文 Word2Vec 词向量训练（gensim）

本 Notebook 使用 `gensim` 在华为云 ModelArts Notebook 中训练 Word2Vec 词向量，
超参数与词向量训练参考配置对齐：

| 超参 | 原值 | gensim 对应 |
|------|------|-------------|
| 模式 | skip-gram（批量） | `sg=1` |
| vector_dim | 100 | `vector_size=100` |
| window_size | 5 | `window=5` |
| min_count | 5 | `min_count=5` |
| epochs | 5 | `epochs=5` |
| sampling_threshold | 0.0001 | `sample=1e-4` |
| negative_samples | 5 | `negative=5` |

训练产物（`vectors.txt` / `vectors.bin`）与原样例的预训练向量格式一致，可直接用于下游分类特征。


## 1. 依赖与数据准备

In [ ]:
# !pip install gensim pandas

import pandas as pd
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

# 读取推文语料（每行一句）
tweets = pd.read_csv('../../data/tweet_telco.csv', encoding='utf-8', header=None, names=['text'])
tweets['text'].to_csv('tweets_corpus.txt', encoding='utf-8', index=False, header=False)
print(f"语料行数: {len(tweets)}")

## 2. 训练 Word2Vec（超参对齐 batch_skipgram）

In [ ]:
# 分词：简单按空白切分（原流程基于推文原始文本，未做复杂清洗）
sentences = [line.split() for line in open('tweets_corpus.txt', encoding='utf-8') if line.strip()]
print(f"句子数: {len(sentences)}")

In [ ]:
model = Word2Vec(
    sentences=sentences,
    sg=1,                # skip-gram（对齐 batch_skipgram）
    vector_size=100,     # 对齐 vector_dim=100
    window=5,            # 对齐 window_size=5
    min_count=5,         # 对齐 min_count=5
    sample=1e-4,         # 对齐 sampling_threshold=0.0001
    negative=5,          # 对齐 negative_samples=5
    epochs=5,            # 对齐 epochs=5
    workers=4,           # 多核
)
print(f"词表大小: {len(model.wv)}")

## 3. 相似度验证

In [ ]:
# 电信业务场景相似词验证
# 业务场景说明：
# - 用于验证词向量模型是否学习到电信行业的语义关系
# - 支持客服系统相似问题推荐、投诉自动分类、热点话题发现等场景
#
# 验证维度：
# 1. 设备品牌词：验证"iphone"与"android"等竞品是否具有语义关联
# 2. 网络制式词：验证"5G"与"4G""LTE"等网络制式的语义关系
# 3. 服务质量词：验证"信号""网络""覆盖"等服务相关词的相似度
# 4. 投诉类型词：验证"投诉""故障""断网"等投诉相关词的语义聚类

# 验证词汇列表（电信行业关键词）
test_words = {
    '设备品牌': ['iphone', 'samsung', 'huawei', 'xiaomi'],
    '网络制式': ['5g', '4g', 'lte', 'network', 'signal'],
    '服务质量': ['coverage', 'signal', 'speed', 'bandwidth', 'service'],
    '投诉类型': ['complaint', 'problem', 'issue', 'disconnect', 'slow']
}

for category, words in test_words.items():
    print(f"\n=== {category} ===")
    for w in words:
        try:
            similar = model.wv.most_similar(w, topn=3)
            print(f"{w}: {similar}")
        except KeyError:
            print(f"{w}: 不在词表中（可能需要补充语料）")

## 4. 保存向量（与原格式兼容）

In [ ]:
# 保存为 word2vec 文本格式（首行: 词数 维度）
model.wv.save_word2vec_format('vectors.txt', binary=False)
# 保存二进制格式（供下游快速加载）
model.wv.save_word2vec_format('vectors.bin', binary=True)
# gensim 原生格式
model.save('word2vec.model')
print("已保存: vectors.txt / vectors.bin / word2vec.model")

# 可选：上传到 OBS
# !obsutil cp ./vectors.bin obs://<bucket>/custom-nlp-pipeline/model/vectors.bin

## 5.（可选）t-SNE 可视化

In [ ]:
# from sklearn.manifold import TSNE
# import matplotlib.pyplot as plt
#
# words = list(model.wv.index_to_key)[:500]
# vecs = model.wv[words]
# tsne = TSNE(n_components=2, random_state=0).fit_transform(vecs)
# plt.figure(figsize=(12,12))
# for i, w in enumerate(words):
#     plt.scatter(tsne[i,0], tsne[i,1])
#     plt.annotate(w, (tsne[i,0], tsne[i,1]))
# plt.show()